# agent-lens — quickstart notebook

**Run the scientific method on your LLM agent.**

State a hypothesis. Fork. Compare. Know if it actually worked.

This notebook shows the full workflow in 5 minutes — **no API key required**.

```
pip install agentlens-tracer
```

---

### What you'll see

1. **Run A** — verbose LLM call (high tokens, slow)
2. **Hypothesis** — "a shorter prompt will produce a concise response"
3. **Run B** — fork with edited prompt (low tokens, fast)
4. **Diff** — `GET /runs/A/diff/B` returns metrics delta + `verdict: "improved"`


In [ ]:
!pip install agentlens-tracer -q

## Step 1 — Start the dashboard

agent-lens spins up a local FastAPI server at `localhost:7878`.
We pass a temporary store so this notebook doesn't touch your real traces.

In [ ]:
import tempfile, time, json, uuid, urllib.request

import agent_lens
from agent_lens.models import Event, EventType, Run, RunStatus, Span
from agent_lens.store import Store

store = Store(path=tempfile.mktemp(suffix=".db"))
agent_lens.dashboard.start(store=store, open_browser=False)
time.sleep(1.0)
print("Dashboard running at http://localhost:7878")

## Step 2 — Run A: verbose agent

We simulate a verbose LLM call: a long system prompt, long response, high token count.

> **With a real API key?** Replace the mock data below with an actual `client.chat.completions.create()` call wrapped in `@agent_lens.trace`.

In [ ]:
QUESTION = "What is Python's GIL?"

VERBOSE_SYSTEM = (
    "You are an expert technical assistant. Provide comprehensive explanations "
    "covering all relevant background, historical context, examples, and edge cases."
)

VERBOSE_RESPONSE = (
    "The Global Interpreter Lock (GIL) in Python is a mutex that protects access to Python "
    "objects, preventing multiple threads from executing Python bytecodes simultaneously. "
    "Historical context: introduced in CPython as a simpler alternative to fine-grained locking. "
    "Impact: the GIL prevents true CPU-level parallelism in threads but allows I/O concurrency. "
    "Workarounds: multiprocessing, C extensions that release the GIL (NumPy/SciPy). "
    "Python 3.13 introduces experimental free-threaded mode via --disable-gil (PEP 703)."
)

now = time.time()
run_a_id, span_a_id = str(uuid.uuid4()), str(uuid.uuid4())
t_a = now - 3.0

store.save_run(Run(id=run_a_id, name="verbose_agent",
                   start_time=t_a, end_time=t_a + 1.847,
                   status=RunStatus.COMPLETED))
store.save_span(Span(id=span_a_id, run_id=run_a_id, name="chat.completions", type="llm",
                     start_time=t_a, end_time=t_a + 1.847))
store.save_event(Event(run_id=run_a_id, span_id=span_a_id, type=EventType.LLM_START,
                       timestamp=t_a,
                       data={"messages": [{"role": "system", "content": VERBOSE_SYSTEM},
                                          {"role": "user",   "content": QUESTION}]}))
store.save_event(Event(run_id=run_a_id, span_id=span_a_id, type=EventType.LLM_END,
                       timestamp=t_a + 1.847,
                       data={"latency_ms": 1847, "total_tokens": 453, "cost_usd": 0.0045,
                             "response": {"choices": [{"message": {"content": VERBOSE_RESPONSE}}]}}))

print(f"Run A: {run_a_id[:8]}...  tokens=453  latency=1847ms  cost=$0.0045")

## Step 3 — State a hypothesis and fork

Before making any change, we write down *why* — a hypothesis and an expected outcome.

This is the key discipline agent-lens adds: **every prompt change is a documented experiment.**

In [ ]:
# The hypothesis
HYPOTHESIS = "A shorter system prompt will produce a concise response"
EXPECTED   = "concise"   # this substring should appear in Run B's response

print(f"Hypothesis : {HYPOTHESIS}")
print(f"Expected   : response contains {EXPECTED!r}")

In [ ]:
CONCISE_SYSTEM = "Answer in one short sentence."

# Note: CONCISE_RESPONSE deliberately contains the word 'concise'
# so the assertion passes. With a real API call you'd test whatever
# quality attribute matters to you.
CONCISE_RESPONSE = (
    "Python's GIL is a concise mutex that ensures only one thread executes Python "
    "bytecode at a time, preventing true parallelism but simplifying memory management."
)

run_b_id, span_b_id = str(uuid.uuid4()), str(uuid.uuid4())
t_b = now - 1.0

store.save_run(Run(id=run_b_id, name="concise_agent",
                   start_time=t_b, end_time=t_b + 0.820,
                   status=RunStatus.COMPLETED,
                   parent_run_id=run_a_id, fork_span_id=span_a_id,
                   notes=HYPOTHESIS, expected_output=EXPECTED))
store.save_span(Span(id=span_b_id, run_id=run_b_id, name="chat.completions", type="llm",
                     start_time=t_b, end_time=t_b + 0.820))
store.save_event(Event(run_id=run_b_id, span_id=span_b_id, type=EventType.LLM_START,
                       timestamp=t_b,
                       data={"messages": [{"role": "system", "content": CONCISE_SYSTEM},
                                          {"role": "user",   "content": QUESTION}]}))
store.save_event(Event(run_id=run_b_id, span_id=span_b_id, type=EventType.LLM_END,
                       timestamp=t_b + 0.820,
                       data={"latency_ms": 820, "total_tokens": 87, "cost_usd": 0.00087,
                             "response": {"choices": [{"message": {"content": CONCISE_RESPONSE}}]}}))

print(f"Run B: {run_b_id[:8]}...  tokens=87  latency=820ms  cost=$0.00087")
print(f"       fork of Run A, notes={HYPOTHESIS!r}")

## Step 4 — GET /runs/A/diff/B

The diff endpoint compares two runs structurally:
- Which messages changed
- Metrics delta (latency, tokens, cost) with % change
- Whether `expected_output` was met → **verdict**

In [ ]:
diff_url = f"http://localhost:7878/runs/{run_a_id}/diff/{run_b_id}"

for attempt in range(15):
    try:
        with urllib.request.urlopen(diff_url, timeout=3) as resp:
            diff = json.loads(resp.read())
        break
    except Exception:
        if attempt == 14: raise
        time.sleep(0.4)

print(json.dumps(diff, indent=2))

## Step 5 — The verdict

In [ ]:
m  = diff["metrics_delta"]
ar = diff["assertion_result"]

print("── Metrics delta ───────────────────────────────────")
for key in ("latency_ms", "total_tokens", "cost_usd"):
    v = m[key]
    print(f"  {key:<14}  {v['a']}  →  {v['b']}  ({v['pct_change']:+.1f}%)")

print("\n── Assertion ───────────────────────────────────────")
print(f"  expected_output : {ar['expected_output']!r}")
print(f"  passed in A     : {ar['passed_in_a']}")
print(f"  passed in B     : {ar['passed_in_b']}")

verdict = ar["verdict"]
emoji   = {"improved": "✅", "regressed": "❌", "both_pass": "✅", "neither_pass": "⚠️"}
print(f"\n  verdict: {emoji.get(verdict, '')} {verdict.upper()}")

## What just happened

```
Run A (verbose)   → 453 tokens, 1847ms, $0.0045
Run B (concise)   →  87 tokens,  820ms, $0.00087

tokens:  -80.8%
latency: -55.6%
cost:    -80.7%

hypothesis: CONFIRMED → verdict: improved
```

The key insight: you wrote down *why* before making the change. The diff confirmed it. Now there's a permanent record of that experiment in your trace database — not just in a Slack message or your memory.

---

## Try it with a real agent

```python
import agent_lens
from openai import OpenAI

agent_lens.install()          # patches OpenAI + Anthropic automatically
agent_lens.dashboard.start()  # http://localhost:7878

client = OpenAI()

@agent_lens.trace
def my_agent(query: str) -> str:
    return client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": query}]
    ).choices[0].message.content

my_agent("Explain Python's GIL")
# Then open the dashboard, click Pause, edit the system prompt, Fork,
# and GET /runs/{a}/diff/{b} to see the verdict.
```

```bash
pip install agentlens-tracer
```

⭐ **[Star agent-lens on GitHub](https://github.com/RAJUSHANIGARAPU/agent-lens)** if this was useful.
